# Pricing Collection and Normalization

Extracts official network tariffs from downloaded Westnetz and Amprion price-sheet PDFs
into a normalized tariff table.

**Every generated customer uses exactly ONE fixed tariff profile** for now.


Amprion is the **transmission** operator, relevant only to customers connected directly at
extra-high voltage. We don't use it for now but will in the future.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "../src")
import pricing as pr

PRICING_DIR = Path("../data/inputs/raw/pricing")
OUTPUTS_DIR = Path("../data/outputs/pricing")
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

WESTNETZ_PDF = PRICING_DIR / "westnetz_2026.pdf"
AMPRION_PDF = PRICING_DIR / "amprion_2026.pdf"

print("Westnetz PDF found:", WESTNETZ_PDF.exists())
print("Amprion PDF found:", AMPRION_PDF.exists())


Westnetz PDF found: True
Amprion PDF found: True


## 1. Extract both operators' annual demand-price tables

In [2]:
westnetz_tariffs = pr.extract_westnetz_annual_tariffs(WESTNETZ_PDF)
display(westnetz_tariffs)


,voltage_level,low_util_leistungspreis_eur_kwa,low_util_arbeitspreis_ct_kwh,high_util_leistungspreis_eur_kwa,high_util_arbeitspreis_ct_kwh
0,Höchstspannung mit Umspannung auf Hochspannung,17.18,2.81,76.70,0.43
1,Hochspannung,13.72,4.57,122.22,0.23
2,Hochspannung mit Umspannung auf Mittelspannung,14.75,4.85,130.25,0.23
3,Mittelspannung,16.82,5.71,134.32,1.01
4,Mittelspannung mit Umspannung auf Niederspannung,17.90,6.57,150.90,1.25
5,Niederspannung,19.93,8.27,120.43,4.25


In [3]:
amprion_tariffs = pr.extract_amprion_annual_tariffs(AMPRION_PDF)
display(amprion_tariffs)


,voltage_level,low_util_leistungspreis_eur_kwa,low_util_arbeitspreis_ct_kwh,high_util_leistungspreis_eur_kwa,high_util_arbeitspreis_ct_kwh
0,Höchstspannung,11.39,2.36,53.06,0.69
1,Höchstspannung mit Umspannung,17.18,2.81,76.70,0.43


## 2. Check

Westnetz's "Höchstspannung mit Umspannung auf Hochspannung" row and Amprion's "Höchstspannung mit
Umspannung" row describe the same physical interconnection point between the distribution and
transmission grids — they should match exactly if extraction is correct.

In [4]:
westnetz_boundary = westnetz_tariffs[
    westnetz_tariffs["voltage_level"] == "Höchstspannung mit Umspannung auf Hochspannung"
].iloc[0]
amprion_boundary = amprion_tariffs[
    amprion_tariffs["voltage_level"] == "Höchstspannung mit Umspannung"
].iloc[0]

price_cols = [c for c in westnetz_tariffs.columns if c != "voltage_level"]
matches = all(westnetz_boundary[c] == amprion_boundary[c] for c in price_cols)
print("Boundary tariffs match between operators:", matches)
assert matches, "Extraction mismatch - re-inspect the PDFs manually."


Boundary tariffs match between operators: True


## 3. Normalizing into one table

In [5]:
network_tariffs = pr.normalize_tariffs(westnetz_tariffs, amprion_tariffs)
display(network_tariffs)


,operator,voltage_level,low_util_leistungspreis_eur_kwa,low_util_arbeitspreis_ct_kwh,high_util_leistungspreis_eur_kwa,high_util_arbeitspreis_ct_kwh
0,westnetz,Höchstspannung mit Umspannung auf Hochspannung,17.18,2.81,76.70,0.43
1,westnetz,Hochspannung,13.72,4.57,122.22,0.23
2,westnetz,Hochspannung mit Umspannung auf Mittelspannung,14.75,4.85,130.25,0.23
3,westnetz,Mittelspannung,16.82,5.71,134.32,1.01
4,westnetz,Mittelspannung mit Umspannung auf Niederspannung,17.90,6.57,150.90,1.25
5,westnetz,Niederspannung,19.93,8.27,120.43,4.25
6,amprion,Höchstspannung,11.39,2.36,53.06,0.69
7,amprion,Höchstspannung mit Umspannung,17.18,2.81,76.70,0.43


## 4. Extract the single tariff profile used by this project

`Mittelspannung mit Umspannung auf Niederspannung` (Westnetz).

In [6]:
default_tariff = pr.get_default_tariff(network_tariffs)
default_tariff


{'operator': 'westnetz',
 'voltage_level': 'Mittelspannung mit Umspannung auf Niederspannung',
 'low_util_leistungspreis_eur_kwa': 17.9,
 'low_util_arbeitspreis_ct_kwh': 6.57,
 'high_util_leistungspreis_eur_kwa': 150.9,
 'high_util_arbeitspreis_ct_kwh': 1.25}

## 5. Save outputs

In [7]:
tariffs_path = OUTPUTS_DIR / "network_tariffs_normalized.csv"
network_tariffs.to_csv(tariffs_path, index=False)

print("Saved:", tariffs_path)


Saved: ..\data\outputs\pricing\network_tariffs_normalized.csv
